# ASL Index & Climatology Workflow

This notebook drives the ASL (Amundsen Sea Low) analysis workflow, which includes:
1. Identifying ASL locations and generating indices from PSL data.
2. Generating seasonal and annual cycle time series (climatology step 1).
3. Calculating mean climatology and bias against observation/reanalysis (climatology step 2).
4. Performing ASL lead-lag regression and correlation analysis.

In [ ]:
import os
import sys
import glob
import collections
import re

# Ensure our shared libraries are discoverable
sys.path.append("..")
from util.common import Case
from util.asl import (
    run_asl_index_generation,
    run_asl_leadlag_analysis,
    run_asl_index_clim_step1,
    run_asl_index_clim_step2
)

In [ ]:
# Initialize Dask Client for Diagnostics Dashboard (Optional)
from dask.distributed import Client
client = Client()
client

In [ ]:
# ============================================================
# Global configuration paths
# ============================================================
top_path = "/lcrc/group/e3sm2/ac.szhang/E3SMv21_testings/v3_polar_paper"
run_path = "/lcrc/group/e3sm/ac.szhang/acme_scratch/data/pcmdi/model/monthly"
run_mask = "/lcrc/group/e3sm/ac.szhang/acme_scratch/data/pcmdi/model/fixed/sftlf"
model_root = "/lcrc/group/e3sm2/ac.szhang/E3SMv21_testings"

# ============================================================
# Shared output-layout definitions
#
# These describe the stable parts of the output directory.
# The final chunk directory, such as 5yr, 10yr, or 30yr, is
# configured separately for each model simulation.
# ============================================================
DATA_LAYOUTS = {
    "atm_ts": {
        "component": "atm",
        "grid": "180x360_aave",
        "product": "ts",
        "frequency": "monthly",
    },
    "atm_clim": {
        "component": "atm",
        "grid": "180x360_aave",
        "product": "clim",
        "frequency": None,
    },
    "lnd_ts": {
        "component": "lnd",
        "grid": "180x360_aave",
        "product": "ts",
        "frequency": "monthly",
    },
    "lnd_clim": {
        "component": "lnd",
        "grid": "180x360_aave",
        "product": "clim",
        "frequency": None,
    },
}


# ============================================================
# Model-case configurations
#
# Set "chunk" to:
#
#   "auto"  -> discover an available chunk directory
#   "5yr"   -> explicitly use the 5yr directory
#   "10yr"  -> explicitly use the 10yr directory
#   "30yr"  -> explicitly use the 30yr directory
#   None    -> use the product directory without a chunk
#
# Historical and SSP370 data from the SORRM histssp370 cases
# are distinguished by the requested analysis period.
# ============================================================

MODEL_CASES = {
    # ========================================================
    # E3SM v2.1 LR historical ensemble
    # ========================================================

    "v2_1_LR_historical_0101": {
        "model_name": "v2_1.LR.historical_0101",
        "case_name": "v2_1-LR",
        "model_version": "v2.1",
        "configuration": "LR",
        "experiment": "historical",
        "member": "0101",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v2_1_LR_historical_0151": {
        "model_name": "v2_1.LR.historical_0151",
        "case_name": "v2_1-LR",
        "model_version": "v2.1",
        "configuration": "LR",
        "experiment": "historical",
        "member": "0151",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v2_1_LR_historical_0201": {
        "model_name": "v2_1.LR.historical_0201",
        "case_name": "v2_1-LR",
        "model_version": "v2.1",
        "configuration": "LR",
        "experiment": "historical",
        "member": "0201",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v2_1_LR_historical_0251": {
        "model_name": "v2_1.LR.historical_0251",
        "case_name": "v2_1-LR",
        "model_version": "v2.1",
        "configuration": "LR",
        "experiment": "historical",
        "member": "0251",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v2_1_LR_historical_0301": {
        "model_name": "v2_1.LR.historical_0301",
        "case_name": "v2_1-LR",
        "model_version": "v2.1",
        "configuration": "LR",
        "experiment": "historical",
        "member": "0301",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v2_1_LR_piControl": {
        "model_name": "v2_1.LR.piControl",
        "case_name": "v2_1-LR",
        "model_version": "v2.1",
        "configuration": "LR",
        "experiment": "piControl",
        "member": None,
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    # ========================================================
    # E3SM v2.1 SORRM
    # ========================================================

    "v2_1_SORRM_control": {
        "model_name": "v2_1.SORRM.control",
        "case_name": "v2_1-SORRM",
        "model_version": "v2.1",
        "configuration": "SORRM",
        "experiment": "piControl",
        "member": None,
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v2_1_SORRM_histssp370_0701": {
        "model_name": "v2_1.SORRM.histssp370_0701",
        "case_name": "v2_1-SORRM",
        "model_version": "v2.1",
        "configuration": "SORRM",
        "experiment": "historical+ssp370",
        "member": "0701",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v2_1_SORRM_histssp370_0751": {
        "model_name": "v2_1.SORRM.histssp370_0751",
        "case_name": "v2_1-SORRM",
        "model_version": "v2.1",
        "configuration": "SORRM",
        "experiment": "historical+ssp370",
        "member": "0751",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v2_1_SORRM_histssp370_0801": {
        "model_name": "v2_1.SORRM.histssp370_0801",
        "case_name": "v2_1-SORRM",
        "model_version": "v2.1",
        "configuration": "SORRM",
        "experiment": "historical+ssp370",
        "member": "0801",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v2_1_SORRM_histssp370_fismf_0701": {
        "model_name": "v2_1.SORRM.histssp370-fismf_0701",
        "case_name": "v2_1-SORRM-fismf",
        "model_version": "v2.1",
        "configuration": "SORRM",
        "experiment": "historical+ssp370",
        "member": "0701",
        "variant": "fismf",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    # ========================================================
    # E3SM v3 LR AMIP ensemble
    # ========================================================

    "v3_LR_amip_0101": {
        "model_name": "v3.LR.amip_0101",
        "case_name": "v3-LR-AMIP",
        "model_version": "v3",
        "configuration": "LR",
        "experiment": "amip",
        "member": "0101",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v3_LR_amip_0151": {
        "model_name": "v3.LR.amip_0151",
        "case_name": "v3-LR-AMIP",
        "model_version": "v3",
        "configuration": "LR",
        "experiment": "amip",
        "member": "0151",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v3_LR_amip_0201": {
        "model_name": "v3.LR.amip_0201",
        "case_name": "v3-LR-AMIP",
        "model_version": "v3",
        "configuration": "LR",
        "experiment": "amip",
        "member": "0201",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    # ========================================================
    # E3SM v3 LR historical ensemble
    # ========================================================

    "v3_LR_historical_0051": {
        "model_name": "v3.LR.historical_0051",
        "case_name": "v3-LR",
        "model_version": "v3",
        "configuration": "LR",
        "experiment": "historical",
        "member": "0051",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v3_LR_historical_0101": {
        "model_name": "v3.LR.historical_0101",
        "case_name": "v3-LR",
        "model_version": "v3",
        "configuration": "LR",
        "experiment": "historical",
        "member": "0101",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v3_LR_historical_0101_bcdt15m": {
        "model_name": "v3.LR.historical_0101_bcdt15m",
        "case_name": "v3-LR-bcdt15m",
        "model_version": "v3",
        "configuration": "LR",
        "experiment": "historical",
        "member": "0101",
        "variant": "bcdt15m",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v3_LR_historical_0151": {
        "model_name": "v3.LR.historical_0151",
        "case_name": "v3-LR",
        "model_version": "v3",
        "configuration": "LR",
        "experiment": "historical",
        "member": "0151",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v3_LR_historical_0201": {
        "model_name": "v3.LR.historical_0201",
        "case_name": "v3-LR",
        "model_version": "v3",
        "configuration": "LR",
        "experiment": "historical",
        "member": "0201",
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    # ========================================================
    # E3SM v3 LR piControl
    # ========================================================

    "v3_LR_piControl": {
        "model_name": "v3.LR.piControl",
        "case_name": "v3-LR",
        "model_version": "v3",
        "configuration": "LR",
        "experiment": "piControl",
        "member": None,
        "variant": "standard",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },

    "v3_LR_piControl_scaled_dismf": {
        "model_name": "v3.LR.piControl-scaled-dismf",
        "case_name": "v3-LR-scaled-dismf",
        "model_version": "v3",
        "configuration": "LR",
        "experiment": "piControl",
        "member": None,
        "variant": "scaled-dismf",
        "outputs": {
            "atm_ts": {"chunk": "auto"},
            "atm_clim": {"chunk": "auto"},
            "lnd_ts": {"chunk": "auto"},
            "lnd_clim": {"chunk": "auto"},
        },
    },
}


# ============================================================
# Analysis periods
#
# The period is separated from MODEL_CASES because multiple
# analysis periods can be selected from the same simulation.
# For example, the SORRM histssp370 simulations contain both
# historical and SSP370 data.
# ============================================================

ANALYSIS_PERIODS = {
    "historical": {
        "start_ym": 195101,
        "end_ym": 201412,
    },
    "ssp370": {
        "start_ym": 201501,
        "end_ym": 210012,
    },
    "historical_ssp370": {
        "start_ym": 195101,
        "end_ym": 210012,
    },
    "historical_1985_2014": {
        "start_ym": 198501,
        "end_ym": 201412,
    },
    "future_2071_2100": {
        "start_ym": 207101,
        "end_ym": 210012,
    },
    "sorrm_piControl": {
        "start_ym": 80101,
        "end_ym": 100012,
    },
}


# ============================================================
# Filename parsing
#
# Supports filenames such as:
#
#   PSL_195101_196012.nc
#   PSL_080101_085012.nc
# ============================================================

_MODEL_FILE_PATTERN = re.compile(
    r"^(?P<variable>.+?)_"
    r"(?P<start>\d{5,6})_"
    r"(?P<end>\d{5,6})\.nc$"
)


def parse_model_filename(path):
    """
    Parse the variable and date range from a model-output filename.

    Returns
    -------
    dict or None
        Dictionary containing path, variable, start_ym, and end_ym.
        Returns None when the filename does not match the expected form.
    """
    basename = os.path.basename(path)
    match = _MODEL_FILE_PATTERN.match(basename)

    if match is None:
        return None

    return {
        "path": path,
        "variable": match.group("variable"),
        "start_ym": int(match.group("start")),
        "end_ym": int(match.group("end")),
    }


def split_ym(ym):
    """Split an integer YYYYMM value into year and month."""
    year, month = divmod(int(ym), 100)

    if month < 1 or month > 12:
        raise ValueError(f"Invalid year-month value: {ym}")

    return year, month


def month_index(ym):
    """Convert YYYYMM to a monotonically increasing month index."""
    year, month = split_ym(ym)
    return year * 12 + month - 1


# ============================================================
# Configuration helpers
# ============================================================

def get_output_config(case_key, output_key):
    """
    Merge the shared layout definition with the case-specific
    output configuration.
    """
    if case_key not in MODEL_CASES:
        valid = ", ".join(sorted(MODEL_CASES))
        raise KeyError(
            f"Unknown model case {case_key!r}. "
            f"Valid cases: {valid}"
        )

    if output_key not in DATA_LAYOUTS:
        valid = ", ".join(sorted(DATA_LAYOUTS))
        raise KeyError(
            f"Unknown output layout {output_key!r}. "
            f"Valid layouts: {valid}"
        )

    case = MODEL_CASES[case_key]
    outputs = case.get("outputs", {})

    if output_key not in outputs:
        valid = ", ".join(sorted(outputs))
        raise KeyError(
            f"Output {output_key!r} is not configured for "
            f"{case_key!r}. Available outputs: {valid}"
        )

    return {
        **DATA_LAYOUTS[output_key],
        **outputs[output_key],
    }


def get_output_base_path(case_key, output_key):
    """
    Construct the output directory without the final chunk directory.
    """
    case = MODEL_CASES[case_key]
    config = get_output_config(case_key, output_key)

    parts = [
        model_root,
        case["model_name"],
        "post",
        config["component"],
        config["grid"],
        config["product"],
    ]

    if config.get("frequency"):
        parts.append(config["frequency"])

    return os.path.join(*parts)


def discover_chunk_directories(base_path):
    """
    Return all available subdirectories under an output product path.

    Directories are sorted by their numeric year length when their
    names follow forms such as 5yr, 10yr, 30yr, or 50yr.
    """
    if not os.path.isdir(base_path):
        return []

    subdirectories = [
        path
        for path in glob.glob(os.path.join(base_path, "*"))
        if os.path.isdir(path)
    ]

    def chunk_sort_key(path):
        name = os.path.basename(path)
        match = re.fullmatch(r"(\d+)yr", name)

        if match:
            return 0, -int(match.group(1))

        return 1, name

    return sorted(subdirectories, key=chunk_sort_key)


def get_model_output_paths(case_key, output_key):
    """
    Return one or more configured output directories.

    When chunk="auto", all available chunk directories are returned.
    Searching all directories allows the date-coverage logic to choose
    the appropriate non-overlapping files.
    """
    config = get_output_config(case_key, output_key)
    base_path = get_output_base_path(case_key, output_key)
    chunk = config.get("chunk")

    if chunk is None:
        return [base_path]

    if chunk == "auto":
        directories = discover_chunk_directories(base_path)

        if not directories:
            raise FileNotFoundError(
                f"No chunk directories found under:\n{base_path}"
            )

        return directories

    output_path = os.path.join(base_path, chunk)

    if not os.path.isdir(output_path):
        raise FileNotFoundError(
            f"Configured output directory does not exist:\n"
            f"{output_path}"
        )

    return [output_path]


def identify_analysis_experiment(case_key, start_ym, end_ym):
    """
    Identify the forcing period represented by the selected dates.

    Combined SORRM historical+SSP370 simulations are classified from
    the selected time range.
    """
    case_experiment = MODEL_CASES[case_key]["experiment"]

    if case_experiment != "historical+ssp370":
        return case_experiment

    if end_ym <= 201412:
        return "historical"

    if start_ym >= 201501:
        return "ssp370"

    return "historical+ssp370"


# ============================================================
# File selection
# ============================================================

def select_continuous_files(candidates, start_ym, end_ym):
    """
    Select a continuous, non-overlapping set of files.

    At each step, the file that covers the next required month and
    extends furthest forward is selected. This avoids simultaneously
    using overlapping 5yr, 10yr, 30yr, or 50yr files.
    """
    candidates = list(candidates)
    selected = []

    current_index = month_index(start_ym)
    final_index = month_index(end_ym)

    while current_index <= final_index:
        available = [
            item
            for item in candidates
            if (
                month_index(item["start_ym"])
                <= current_index
                <= month_index(item["end_ym"])
            )
        ]

        if not available:
            missing_year = current_index // 12
            missing_month = current_index % 12 + 1

            raise RuntimeError(
                "Gap detected in model-output coverage at "
                f"{missing_year:04d}-{missing_month:02d}"
            )

        best = max(
            available,
            key=lambda item: (
                month_index(item["end_ym"]),
                -month_index(item["start_ym"]),
            ),
        )

        selected.append(best)
        current_index = month_index(best["end_ym"]) + 1

    unique = []
    seen_paths = set()

    for item in selected:
        if item["path"] not in seen_paths:
            unique.append(item)
            seen_paths.add(item["path"])

    return unique


def get_model_files(
    case_key,
    output_key="atm_ts",
    variable="PSL",
    start_ym=None,
    end_ym=None,
):
    """
    Return sorted, continuous, non-overlapping model files.

    Parameters
    ----------
    case_key : str
        Key in MODEL_CASES.

    output_key : str
        Output type, such as "atm_ts", "atm_clim",
        "lnd_ts", or "lnd_clim".

    variable : str
        Variable filename prefix, such as "PSL", "TREFHT",
        or "LANDFRAC".

    start_ym, end_ym : int, optional
        Requested time range in YYYYMM format.

        When omitted, all matching files are returned. When supplied,
        only a continuous set covering the requested period is returned.

    Returns
    -------
    list[str]
        Selected model-output file paths.
    """
    output_paths = get_model_output_paths(
        case_key=case_key,
        output_key=output_key,
    )

    files = []

    for output_path in output_paths:
        pattern = os.path.join(output_path, f"{variable}_*.nc")
        files.extend(glob.glob(pattern))

    files = sorted(set(files))

    if not files:
        searched_paths = "\n".join(f"  - {path}" for path in output_paths)

        raise FileNotFoundError(
            f"No files found for variable {variable!r}.\n"
            f"Searched directories:\n{searched_paths}"
        )

    if start_ym is None and end_ym is None:
        return files

    if start_ym is None or end_ym is None:
        raise ValueError(
            "start_ym and end_ym must either both be supplied "
            "or both be omitted."
        )

    if month_index(start_ym) > month_index(end_ym):
        raise ValueError(
            f"start_ym {start_ym} occurs after end_ym {end_ym}"
        )

    candidates = []

    for path in files:
        parsed = parse_model_filename(path)

        if parsed is None:
            continue

        if parsed["variable"] != variable:
            continue

        if (
            month_index(parsed["end_ym"]) >= month_index(start_ym)
            and month_index(parsed["start_ym"]) <= month_index(end_ym)
        ):
            candidates.append(parsed)

    if not candidates:
        raise FileNotFoundError(
            f"No {variable!r} files overlap the requested period "
            f"{start_ym:06d}-{end_ym:06d}"
        )

    selected = select_continuous_files(
        candidates=candidates,
        start_ym=start_ym,
        end_ym=end_ym,
    )

    return [item["path"] for item in selected]


# ============================================================
# Model-selection helper
# ============================================================

def select_model_cases(
    model_version=None,
    configuration=None,
    experiment=None,
    member=None,
    variant=None,
):
    """
    Select model cases using configuration metadata.

    Each argument is optional. Supplied arguments are combined using
    logical AND.
    """
    selected = []

    for case_key, case in MODEL_CASES.items():
        if (
            model_version is not None
            and case["model_version"] != model_version
        ):
            continue

        if (
            configuration is not None
            and case["configuration"] != configuration
        ):
            continue

        if (
            experiment is not None
            and case["experiment"] != experiment
        ):
            continue

        if member is not None and case["member"] != member:
            continue

        if variant is not None and case["variant"] != variant:
            continue

        selected.append(case_key)

    return selected


# ============================================================
# Example model and period selection
# ============================================================

selected_case = "v2_1_SORRM_histssp370_0701"
selected_output = "atm_ts"
selected_variable = "PSL"
selected_period = "ssp370"

period_config = ANALYSIS_PERIODS[selected_period]

start_ym = period_config["start_ym"]
end_ym = period_config["end_ym"]

selected_experiment = identify_analysis_experiment(
    case_key=selected_case,
    start_ym=start_ym,
    end_ym=end_ym,
)

model_files = get_model_files(
    case_key=selected_case,
    output_key=selected_output,
    variable=selected_variable,
    start_ym=start_ym,
    end_ym=end_ym,
)

print(f"Selected case       : {selected_case}")
print(f"Simulation directory: {MODEL_CASES[selected_case]['model_name']}")
print(f"Selected experiment : {selected_experiment}")
print(f"Selected period     : {start_ym:06d}-{end_ym:06d}")
print(f"Selected output     : {selected_output}")
print(f"Number of files     : {len(model_files)}")

for path in model_files:
    print(f"  {path}")


# ============================================================
# Amundsen Sea Low configuration
# ============================================================

asl_region = {
    "west": 170.0,
    "east": 298.0,
    "south": -80.0,
    "north": -60.0,
}

asl_min_dist = 5
asl_num_peak = 3
asl_exc_bord = False
l_check_asl_region = False
l_allow_no_asl = False

## 1. Run ASL Index Generation

In [ ]:
# Define out and figure directories
out_path = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/polar_analysis/data/asl_analysis/raw_index"
fig_path = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/polar_analysis/figure/asl_analysis/asl_index_ts"

# 1.1 Historical E3SM run
case_dict = collections.OrderedDict()
hist_cfg = model_cases["historical"]
sorrm_files = get_model_files(os.path.join(model_root, hist_cfg["model_name"]))
case_dict[hist_cfg["case_name"]] = Case(sorrm_files, "PSL", "blue", hist_cfg["case_name"])
mask_file = os.path.join(run_mask, f"Amon/e3sm.historical.{hist_cfg['case_name']}.fx.sftlf.nc")
if os.path.exists(mask_file):
    case_dict['mask'] = Case(mask_file, "sftlf", "blue", hist_cfg["case_name"])
run_asl_index_generation(fig_path, out_path, "e3sm", "historical", hist_cfg["relm"], "asl_scotthoskingv3", hist_cfg["period"], 
                         case_dict, asl_region, 1, 12, asl_exc_bord, l_check_asl_region, l_allow_no_asl)

# 1.2 Control E3SM run
case_dict = collections.OrderedDict()
ctrl_cfg = model_cases["piControl"]
control_files = get_model_files(os.path.join(model_root, ctrl_cfg["model_name"]))
case_dict[ctrl_cfg["case_name"]] = Case(control_files, "PSL", "blue", ctrl_cfg["case_name"])
mask_file = os.path.join(run_mask, f"Amon/e3sm.piControl.{ctrl_cfg['case_name']}.fx.sftlf.nc")
if os.path.exists(mask_file):
    case_dict['mask'] = Case(mask_file, "sftlf", "blue", ctrl_cfg["case_name"])
run_asl_index_generation(fig_path, out_path, "e3sm", "piControl", ctrl_cfg["relm"], "asl_scotthoskingv3", ctrl_cfg["period"], 
                         case_dict, asl_region, 1, 12, asl_exc_bord, l_check_asl_region, l_allow_no_asl)

# 1.3 Future E3SM SSP370 run
case_dict = collections.OrderedDict()
ssp_cfg = model_cases["ssp370"]
ssp370_files = get_model_files(os.path.join(model_root, ssp_cfg["model_name"]))
case_dict[ssp_cfg["case_name"]] = Case(ssp370_files, "PSL", "blue", ssp_cfg["case_name"])
mask_file = os.path.join(run_mask, f"Amon/e3sm.ssp370.{ssp_cfg['case_name']}.fx.sftlf.nc")
if os.path.exists(mask_file):
    case_dict['mask'] = Case(mask_file, "sftlf", "blue", ssp_cfg["case_name"])
run_asl_index_generation(fig_path, out_path, "e3sm", "ssp370", ssp_cfg["relm"], "asl_scotthoskingv3", ssp_cfg["period"], 
                         case_dict, asl_region, 1, 12, asl_exc_bord, l_check_asl_region, l_allow_no_asl)

# 1.4 Observation/Analysis Runs (NOAA_20C and ERA5)
for obs_name in ["NOAA_20C", "ERA5"]:
    case_dict = collections.OrderedDict()
    obs_file = sorted(glob.glob(os.path.join(run_path, "analysis", obs_name, "Amon/psl/analysis.historical.{}.en00.*.nc".format(obs_name))))[0]
    case_dict[obs_name] = Case(obs_file, "PSL" if obs_name == "NOAA_20C" else "psl", "blue", obs_name)
    mask_file = os.path.join(run_mask, "Amon/analysis.historical.{}.fx.sftlf.nc".format(obs_name))
    case_dict['mask'] = Case(mask_file, "sftlf", "blue", obs_name)
    run_asl_index_generation(fig_path, out_path, "analysis", "historical", "en00", "asl_scotthoskingv3", "195001-201412", 
                             case_dict, asl_region, 1, 12, asl_exc_bord, l_check_asl_region, l_allow_no_asl)

## 2. Run ASL Climatology Index Calculations (Step 1 & Step 2)

In [ ]:
# Directories for Climatology
raw_index_path = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/polar_analysis/data/asl_analysis/raw_index"
ts_index_path = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/polar_analysis/data/asl_analysis/ts_index"
clim_index_path = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/polar_analysis/data/asl_analysis/clim_index"

# Step 1: Calculate Seasonal/Annual Cycle
mips = ["analysis", "cmip6", "e3sm"]
exps = ["historical", "ssp370", "piControl"]
ver = "asl_scotthoskingv3"
seasons = ['ANN', 'DJF', 'JJA', 'MAM', 'SON', 'AC', 'Monthly']
indices = ['lon', 'lat', 'ActCenPres', 'SectorPres', 'RelCenPres']

run_asl_index_clim_step1(mips, exps, ver, seasons, indices, raw_index_path, ts_index_path)

# Step 2: Compare against Observation Climatologies
obs_sets = ['NOAA_20C.en00', 'ERA5.en00']
obs_mips = ['analysis', 'analysis']
periods = ["1950-2014", "1979-2014"]
test_mips = ["cmip6", "e3sm"]
test_exps = ["historical"]
clim_seasons = ['ANN', 'DJF', 'JJA', 'MAM', 'SON', 'AC']

run_asl_index_clim_step2(obs_sets, obs_mips, test_mips, test_exps, periods, clim_seasons, indices, ts_index_path, clim_index_path)

## 3. Run ASL Lead-Lag Analysis

In [ ]:
# Directories for Lead-Lag
lead_lag_out = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/polar_analysis/data/asl_analysis/lead_lag"
lead_lag_fig = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/polar_analysis/figure/asl_analysis/lead_lag"

# E3SM Historical
case_dict = collections.OrderedDict()
case_dict["v2_1-SORRM"] = Case(sorrm_files, "PSL", "blue", "v2_1-SORRM")
run_asl_leadlag_analysis(lead_lag_fig, lead_lag_out, "e3sm", "historical", "0701", "asl_scotthoskingv3", "195101-201412", 
                         case_dict, asl_region, "RelCenPres", l_check_asl_region)